# Mixed-grid aquaplanet

This notebook couples components that live on different horizontal grids: a
slab ocean and slab sea ice on their own displaced-pole grid, and the SPEEDY
T31L8 atmosphere on its Gaussian grid, exchanging fields through ESMF
regridding weights. It builds the coupled model directly in Python, runs it for
30 days, and maps the atmosphere's humidity and the ocean's temperature, each
on its own grid. On a laptop CPU it takes about 2 minutes. The same model, as
one command:

```bash
python -m jem.main +configuration=aquaplanet-slab-mixed-grid
```

This notebook builds directly rather than going through
`jem.configurations.load("aquaplanet-slab-mixed-grid")` -- the shortcut the
notebooks that run a shipped configuration as-is use --
because what it exists to teach is exactly how a mixed-grid coupling is put
together: the ocean and sea ice on the packaged `DisplacedPoleGrid` SCRIP grid
(its pole sits over land, avoiding the polar singularity), and the ESMF weight
files that let it exchange with the atmosphere's own T31 grid. Four weight
files are loaded (one per direction and per algorithm), and the standard
coupling uses three of them; the build cell says which. `jem.exchangers.default_exchanges`'s own
docstring says this split -- conservative maps for the fluxes, so their
budgets survive the interface, bilinear for the intensive sea surface
temperature -- is "the one the mixed-grid example makes by hand", and this
notebook is that example. Its atmosphere is built exactly like
`01_aquaplanet.ipynb`'s, including the explicit `time_step=12` (minutes) --
see that notebook for why it has to be explicit; this configuration
inherits `aquaplanet-slab`'s own atmosphere group unchanged, so the same
12-minute step applies here too. Verified by
`tests/unit/test_notebook_equivalence.py`, `dt` included.

## Build it

In [ ]:
from pathlib import Path

from jem import plot

output_dir = (Path("output") / "01-04_mixed_grid_aquaplanet").resolve()
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
import jax_datetime as jdt
import jcm
import xarray as xr
from jcm.physics.speedy.speedy_coords import get_speedy_coords

from jem import Coupler, default_exchangers, run_chunked
from jem.components import JCMComponent, SlabOceanModel, SlabSeaiceModel
from jem.components.slab import SlabGrid
from jem.config import package_data_path
from jem.regrid import ESMFRegridders

start_date = jdt.to_datetime("2000-01-01")
coupling_timestep = jdt.to_timedelta(1, "day")

# The atmosphere: the same SPEEDY T31L8 aquaplanet as `01_aquaplanet.ipynb`,
# including its explicit `time_step=12` (minutes) -- see that notebook's
# construction cell for why it has to be given explicitly (this
# configuration's atmosphere group is unchanged from `aquaplanet-slab`'s).
atm_model = jcm.model.Model(
    coords=get_speedy_coords(), start_time=start_date, time_step=12
)
atm = JCMComponent(atm_model)


def _land_fraction(path):
    """Return a packaged mask file's `(time, lat, lon)` field as `SlabGrid`'s `(lon, lat)`.

    Takes the first (and only) time record and transposes it onto the layout
    every grid and component in JAX-ESM uses. `+configuration=aquaplanet-slab-
    mixed-grid` reads its mask file the same way.
    """
    return xr.open_dataset(path)["lsm"].to_numpy()[0].transpose()


# The ocean and sea ice run on their OWN grid: a displaced-pole SCRIP grid
# whose pole sits over land, which is how an ocean model avoids the polar
# singularity a shared lon/lat grid would put right at it. The fractional
# mask comes from the packaged land-fraction file (not the SCRIP file's own
# binary `grid_imask`), so a coastal cell keeps its land fraction: the ocean
# integrates every cell with any sea in it (`SlabGrid.ocean_mask`).
grid = SlabGrid.from_scrip(
    package_data_path("jem.data", "DisplacedPoleGrid.SCRIP.nc"),
    fractional_mask=_land_fraction(package_data_path(
        "jem.data", "landsea_mask_fraction_DisplacedPoleGrid.nc")),
)

# The four ESMF weight files mapping between the atmosphere's T31 grid and
# the ocean's displaced-pole grid, one per direction and per algorithm.
# `ESMFRegridders` takes the full set, but the standard coupling table below
# uses three of them: a2o_conserve (the atmosphere's heat flux onto the ocean),
# o2a_conserve (the ice fraction onto the atmosphere) and o2a_bilinear (the
# SST onto the atmosphere). No row carries an atmosphere-to-ocean *state*, so
# a2o_bilinear is loaded and never called.
regridders = ESMFRegridders(
    a2o_conserve=package_data_path(
        "jem.data", "weight_algo-conserve_JCM_T31_to_DisplacedPoleGrid.nc"),
    a2o_bilinear=package_data_path(
        "jem.data", "weight_algo-bilinear_JCM_T31_to_DisplacedPoleGrid.nc"),
    o2a_conserve=package_data_path(
        "jem.data", "weight_algo-conserve_DisplacedPoleGrid_to_JCM_T31.nc"),
    o2a_bilinear=package_data_path(
        "jem.data", "weight_algo-bilinear_DisplacedPoleGrid_to_JCM_T31.nc"),
)

components = {
    "atm": atm,
    "ocn": SlabOceanModel(grid),
    "seaice": SlabSeaiceModel(grid, name="seaice"),
}
# `regrid=` keys are "<direction>_<kind>": "a2o"/"o2a" crossed with the row's
# own "flux"/"state" -- extensive quantities (fluxes, the ice fraction) go
# through the conservative map, the intensive SST through the bilinear one.
# "a2o_state" is accepted for completeness; no standard row uses it.
exchangers = default_exchangers(components, regrid={
    "a2o_flux": regridders["a2o_conserve"],
    "a2o_state": regridders["a2o_bilinear"],
    "o2a_flux": regridders["o2a_conserve"],
    "o2a_state": regridders["o2a_bilinear"],
})
coupler = Coupler(
    components,
    exchangers,
    coupling_timestep=coupling_timestep,
    start_date=start_date,
)
print(repr(coupler))

## Run it

In [ ]:
result = run_chunked(
    coupler,
    total_time="30 days",
    chunk="30 days",
    output_dir=str(output_dir),
    subsample=3,       # 10 records out of 30 coupled days
    checkpoint_path=None,
)
result.steps_completed, [p.name for p in result.paths]

## What it wrote

One file per component per chunk, named after the coupled step its chunk starts on (`<component>-<first step>.nc`).

In [ ]:
atm_ds = plot.open_output(output_dir, "atm")
ocn_ds = plot.open_output(output_dir, "ocn")
list(ocn_ds.data_vars)

## Plot

Only the ocean's active cells are drawn on the displaced-pole map: a cell that is entirely land is never integrated and holds a constant placeholder temperature (the 288.15 K initial value), which would otherwise show up as a patch of "ocean" at that temperature, such as over North Africa. The mask is the grid's own `ocean_mask`, the one the ocean integrates with.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# The atmosphere's field has 1-D lat/lon (its own T31 grid); the
# ocean's has 2-D lat/lon over the displaced-pole grid's own index
# dimensions -- map_plot draws each on its own grid, unregridded.
# `level` is the atmosphere's sigma coordinate, surface-first, so the level
# nearest sigma = 1 is the lowest model level; the title reads it back.
humidity = atm_ds["specific_humidity"].sel(level=1.0, method="nearest").isel(time=-1)
plot.map_plot(humidity, ax=axes[0],
              title=("Atmosphere: specific humidity, lowest model level "
                     f"(sigma = {float(humidity.level):.2f}) [kg/kg]"))

# Mask the ocean field with the grid's own ocean mask rather than a value
# test, so it stays right whichever cells a configuration leaves inactive.
sst = ocn_ds["sea_surface_temperature"].isel(time=-1) - 273.15
ocean = xr.DataArray(np.asarray(grid.ocean_mask, dtype=bool), dims=sst.dims)
plot.map_plot(sst.where(ocean), ax=axes[1],
              title="Ocean: sea surface temperature [°C] (displaced-pole grid)")
plt.tight_layout()
